In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [2]:
import subprocess
import sys

TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
)

print("Official packages installed successfully")

installing: transformers==4.46.* accelerate==1.1.*
Official packages installed successfully


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="cuda",
)

print("Model loaded successfully")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully


In [4]:
import time
import threading
from transformers import TextIteratorStreamer

def prompt_of_len(n_tokens: int) -> str:
    base = "Explain the following in detail.\n"
    filler = "A data center serves many inference requests at once. " * 600
    ids = tok(base + filler)["input_ids"][:n_tokens]
    return tok.decode(ids)

def measure_stream(prompt: str, new_tokens: int = 128):
    enc = tok(prompt, return_tensors="pt").to("cuda")

    streamer = TextIteratorStreamer(
        tok,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    kwargs = dict(
        **enc,
        max_new_tokens=new_tokens,
        do_sample=False,
        streamer=streamer,
    )

    th = threading.Thread(
        target=model.generate,
        kwargs=kwargs,
    )

    t0 = time.time()
    th.start()

    stamps = []
    for _ in streamer:
        stamps.append(time.time())

    th.join()

    ttft = stamps[0] - t0

    if len(stamps) > 1:
        gaps = [b - a for a, b in zip(stamps, stamps[1:])]
        tpot = sum(gaps) / len(gaps)
    else:
        tpot = 0.0

    total = stamps[-1] - t0

    return {
        "ttft_s": round(ttft, 4),
        "tpot_s": round(tpot, 4),
        "total_s": round(total, 4),
        "n_tokens": len(stamps),
    }

print("Measurement functions are ready")

Measurement functions are ready


In [5]:
warmup_result = measure_stream(
    prompt_of_len(128),
    new_tokens=8
)

print(warmup_result)
print("Warm-up completed")

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


{'ttft_s': 1.1069, 'tpot_s': 0.0433, 'total_s': 1.4529, 'n_tokens': 9}
Warm-up completed


In [6]:
ttft_by_len = {}

for n in [128, 512, 2048]:
    r = measure_stream(prompt_of_len(n))
    ttft_by_len[str(n)] = r["ttft_s"]
    print(n, r)

print("TTFT measurements completed")

128 {'ttft_s': 0.0437, 'tpot_s': 0.0311, 'total_s': 4.022, 'n_tokens': 129}
512 {'ttft_s': 0.0634, 'tpot_s': 0.0314, 'total_s': 4.0862, 'n_tokens': 129}
2048 {'ttft_s': 0.314, 'tpot_s': 0.0363, 'total_s': 4.9666, 'n_tokens': 129}
TTFT measurements completed


In [7]:
import gc

def kv_formula_kb_per_token(
    layers=28,
    kv_heads=2,
    head_dim=128,
    dbytes=2,
):
    return 2 * layers * kv_heads * head_dim * dbytes / 1024

def cache_bytes(pkv):
    if hasattr(pkv, "key_cache"):
        tensors = list(pkv.key_cache) + list(pkv.value_cache)
    else:
        tensors = [t for layer in pkv for t in layer]

    return sum(
        t.numel() * t.element_size()
        for t in tensors
    )

def measure_kv(context: int, new_tokens: int = 256):
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    enc = tok(
        prompt_of_len(context),
        return_tensors="pt",
    ).to("cuda")

    before = torch.cuda.memory_allocated()

    out = model.generate(
        **enc,
        max_new_tokens=new_tokens,
        do_sample=False,
        use_cache=True,
        return_dict_in_generate=True,
    )

    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()
    total_tokens = out.sequences.shape[1]

    return {
        "context": context,
        "total_tokens": int(total_tokens),
        "peak_kb_per_token": round(
            (peak - before) / total_tokens / 1024,
            1,
        ),
        "kv_kb_per_token": round(
            cache_bytes(out.past_key_values) / total_tokens / 1024,
            1,
        ),
    }

print("KV measurement functions are ready")

KV measurement functions are ready


In [8]:
formula = kv_formula_kb_per_token()
print("formula KB/token:", formula)

kv_rows = [measure_kv(c) for c in [512, 2048, 4096]]

for r in kv_rows:
    print(r, "vs formula", formula, "KB/token")

formula KB/token: 28.0


From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


{'context': 512, 'total_tokens': 768, 'peak_kb_per_token': 63.4, 'kv_kb_per_token': 28.0} vs formula 28.0 KB/token
{'context': 2048, 'total_tokens': 2304, 'peak_kb_per_token': 84.0, 'kv_kb_per_token': 28.0} vs formula 28.0 KB/token
{'context': 4096, 'total_tokens': 4352, 'peak_kb_per_token': 87.6, 'kv_kb_per_token': 28.0} vs formula 28.0 KB/token


In [9]:
import json

with open("kv_check.json", "w") as f:
    json.dump(
        {
            "formula_kb_per_token": formula,
            "measured_kb_per_token": kv_rows[-1]["kv_kb_per_token"],
            "peak_kb_per_token": kv_rows[-1]["peak_kb_per_token"],
        },
        f,
        indent=2,
    )

print("wrote kv_check.json")

wrote kv_check.json


In [10]:
QUEUE = [32, 32, 32, 256] * 6

def static_queue(
    batch: int,
    prompt: str = "Explain what an inference server does.",
):
    t0 = time.time()
    useful = 0
    slots = 0

    for i in range(0, len(QUEUE), batch):
        chunk = QUEUE[i:i + batch]
        n = max(chunk)

        enc = tok(
            [prompt] * len(chunk),
            return_tensors="pt",
            padding=True,
        ).to("cuda")

        model.generate(
            **enc,
            max_new_tokens=n,
            do_sample=False,
        )

        useful += sum(chunk)
        slots += n * len(chunk)

    dt = time.time() - t0

    return {
        "batch": batch,
        "wall_s": round(dt, 2),
        "tokens_per_s": round(useful / dt, 1),
        "slot_efficiency": round(useful / slots, 3),
    }

print("Static batcher is ready")

Static batcher is ready


In [11]:
batch_rows = {}

for n in [1, 4, 8]:
    r = static_queue(n)
    batch_rows[str(n)] = r["tokens_per_s"]
    print(r)

print("Batch measurements completed")

{'batch': 1, 'wall_s': 57.76, 'tokens_per_s': 36.6, 'slot_efficiency': 1.0}
{'batch': 4, 'wall_s': 38.68, 'tokens_per_s': 54.6, 'slot_efficiency': 0.344}
{'batch': 8, 'wall_s': 19.87, 'tokens_per_s': 106.3, 'slot_efficiency': 0.344}
Batch measurements completed


In [12]:
baselines = {
    "model": MODEL,
    "dtype": "fp16",
    "ttft_s": ttft_by_len,
    "tpot_s": measure_stream(
        prompt_of_len(512)
    )["tpot_s"],
    "batch": {
        k: v
        for k, v in batch_rows.items()
    },
}

with open("baselines.json", "w") as f:
    json.dump(baselines, f, indent=2)

print(json.dumps(baselines, indent=2))

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "dtype": "fp16",
  "ttft_s": {
    "128": 0.0437,
    "512": 0.0634,
    "2048": 0.314
  },
  "tpot_s": 0.036,
  "batch": {
    "1": 36.6,
    "4": 54.6,
    "8": 106.3
  }
}


In [13]:
from google.colab import files

files.download("baselines.json")
files.download("kv_check.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
from google.colab import files
uploaded = files.upload()

Saving verify_cell (4).py to verify_cell (4).py


In [15]:
!python "/content/verify_cell (4).py"

ttft lengths: ['128', '2048', '512'], tpot_s: 0.036
batch tokens/s 1/4/8: 36.6/54.6/106.3
KV measured 28.0 KB/token vs formula 28.0 KB/token
GREEN CHECK: PASS
